In [1]:
#run this
from rdkit import Chem
import pathlib
import polaris as po
import fastpdb
import datamol as dm

from tempfile import TemporaryDirectory
import fsspec
import zipfile

#competition = po.load_competition("asap-discovery/antiviral-ligand-poses-2025")

/home/campus.ncl.ac.uk/c2049423/anaconda3/envs/polaris/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
competition.cache()

In [ ]:
train, test = competition.get_train_test_split()

In [ ]:
competition.split

In [ ]:
import zipfile
# download the ref files so we can align to them
with fsspec.open("https://fs.polarishub.io/2025-01-asap-discovery/ligand_poses_reference_structures.zip") as fd:
    with zipfile.ZipFile(fd, 'r') as zip_ref:
        zip_ref.extractall("./reference_structures/")

In [ ]:
import pymol
from rdkit import Chem
from rdkit.Chem import AllChem

def align_training_complex(training_molecule, folder):
    # get the smiles of the training molecule
    smiles = Chem.CanonSmiles(Chem.MolToSmiles(training_molecule))
    best_pose_mol = Chem.MolFromMolFile(str(folder/"best_pose.sdf"), removeHs=False)
    if best_pose_mol is None:
        return False
    
    # look up the complex structure in the competition and write it out
    training_datapoint = None
    for i, train_point in enumerate(train):
        if smiles == train_point[0]["CXSMILES"]:
            print(train_point)
            datapoint = competition[i]
            training_datapoint = datapoint
            break
            
    if training_datapoint is None:
        print("could not find datapoint")
        
    out_file = fastpdb.PDBFile()
    out_file.set_structure(training_datapoint["Complex Structure"])
    out_file.write(folder / "complex_template.pdb")
    
    # taken from the example
    # Let's reset! Just to be sure.
    pymol.cmd.delete("all")

    # Load the reference and mobile structure
    ref_path = "./reference_structures/SARS-CoV-2-Mpro/complex.pdb"
    pymol.cmd.load(ref_path, "reference")
    pymol.cmd.load(folder / "complex_template.pdb", "mobile")

    # Compute the RMSD
    pymol.cmd.rms_cur("mobile", "reference")
    # Now let's align our proteins again using PyMol
    pymol.cmd.align(
        "polymer and name CA and mobile",
        "polymer and name CA and reference",
        quiet=0,
    )
    # write out the aligned structure
    pymol.cmd.select("lig", "resn UNK and mobile")
    ret_path = folder / "aligned_template.sdf"
    pymol.cmd.save(ret_path, "lig")
    aligned_temp_mol = Chem.MolFromMolFile(str(folder/"aligned_template.sdf"))
    # workout the transform between the training mol and the aligned version
    rotation_trans = AllChem.GetAlignmentTransform(training_molecule, aligned_temp_mol)[1]
    AllChem.TransformConformer(best_pose_mol.GetConformer(), rotation_trans)
    Chem.MolToMolFile(best_pose_mol, str(folder/"aligned_best_pose.sdf"))
    return True

In [ ]:
for base_folder in ["outputs_fails"]:
    b_folder = pathlib.Path(base_folder)
    for lig_folder in b_folder.glob("*"):
        if "ligand" in lig_folder.name:
            if lig_folder.joinpath("aligned_best_pose.sdf").exists():
                continue
            print("running ", lig_folder)
            mol = Chem.MolFromMolFile(str(lig_folder / "template_lig.sdf"))
            result = align_training_complex(mol, lig_folder)
            print(lig_folder, result)

In [2]:
# run this
# build a list of best pose ligands directly from outputs/

from rdkit import Chem
import pathlib

best_poses_by_smiles = {}

for base_folder in ["outputs"]:
    b_folder = pathlib.Path(base_folder)

    for lig_folder in b_folder.glob("ligand_*"):

        best_pose_file = lig_folder / "best_pose.sdf"

        if best_pose_file.exists():

            mol = Chem.MolFromMolFile(str(best_pose_file), removeHs=False)

            if mol is None:
                print(f"Could not read {best_pose_file}")
                continue

            smiles = Chem.CanonSmiles(Chem.MolToSmiles(mol))

            best_poses_by_smiles[smiles] = mol

print("Loaded poses:", len(best_poses_by_smiles))

Loaded poses: 83


In [3]:
# run this
len(best_poses_by_smiles)

83

In [5]:
# run this
from rdkit import Chem
from rdkit.Chem import rdMolAlign
from rdkit.Chem.rdFMCS import FindMCS

# load reference ligand
ref = Chem.MolFromMolFile(
    "reference_structures/SARS-CoV-2-Mpro/ligand.sdf",
    removeHs=False
)

if ref is None:
    raise ValueError("Could not load reference ligand")

aligned_mols = []

for smiles, mol in best_poses_by_smiles.items():

    try:

        # find maximum common substructure
        mcs = FindMCS(
            [mol, ref],
            ringMatchesRingOnly=True,
            completeRingsOnly=True,
            timeout=10
        )

        if mcs.numAtoms < 6:
            print(f"MCS too small for {smiles}")
            continue

        patt = Chem.MolFromSmarts(mcs.smartsString)

        mol_match = mol.GetSubstructMatch(patt)
        ref_match = ref.GetSubstructMatch(patt)

        if not mol_match or not ref_match:
            print(f"No MCS match for {smiles}")
            continue

        atom_map = list(zip(mol_match, ref_match))

        # make a copy
        mol_aligned = Chem.Mol(mol)

        rmsd = rdMolAlign.AlignMol(
            mol_aligned,
            ref,
            atomMap=atom_map
        )

        mol_aligned.SetProp("RMSD", str(rmsd))

        aligned_mols.append(mol_aligned)

    except Exception as e:
        print(f"Failed for {smiles}: {e}")

print("Aligned molecules:", len(aligned_mols))

MCS too small for CCS(=O)(=O)N1CC2(C1)C(=O)N(c1cncc3ccccc13)C(=O)N2C
Aligned molecules: 82


In [ ]:
# workout if we have any missing ligands
missing_entries = []
for t in test:
    if t["Protein Label"] == "SARS-CoV-2 Mpro":
        if Chem.CanonSmiles(t["CXSMILES"]) not in best_poses_by_smiles:
            missing_entries.append(t)

In [ ]:
Chem.MolFromSmiles(missing_entries[10]["CXSMILES"])

In [6]:
# run this
writer = Chem.SDWriter("best_sars.sdf")

for mol in aligned_mols:
    writer.write(mol)

writer.close()

print("Wrote aligned best_sars.sdf")

Wrote aligned best_sars.sdf
